In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
from pathlib import Path

In [12]:
class Video3DCNN(nn.Module):
    def __init__(self, num_classes=3, in_channels=1):
        super(Video3DCNN, self).__init__()
        
        # First 3D convolutional layer
        self.conv1 = nn.Conv3d(in_channels, 16, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn1 = nn.BatchNorm3d(16)
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2))  # Don't pool in time dimension
        
        # Second 3D convolutional layer
        self.conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn2 = nn.BatchNorm3d(32)
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2))  # Pool in all dimensions
        
        # Third 3D convolutional layer
        self.conv3 = nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn3 = nn.BatchNorm3d(64)
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2))
        
        # Adaptive pooling to handle variable input sizes
        self.adaptive_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        
        # Fully connected layers
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # Input shape: (batch_size, channels, frames, height, width)
        
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        
        # Global pooling
        x = self.adaptive_pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

def load_video_frames(video_path, max_frames=32, grayscale=True):
    """
    Load video frames and prepare them for the 3D CNN
    """
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            # Add channel dimension: (height, width) -> (height, width, 1)
            frame = frame[:, :, np.newaxis]
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = frame.astype(np.float32) / 255.0
        
        # Resize to a fixed size
        frame = cv2.resize(frame, (112, 112))
        frames.append(frame)
    
    cap.release()
    
    # If we have fewer frames, pad with zeros
    if len(frames) < max_frames:
        padding = max_frames - len(frames)
        frames.extend([np.zeros_like(frames[0])] * padding)
    
    # Convert to tensor: (frames, height, width, channels) -> (channels, frames, height, width)
    video_tensor = torch.from_numpy(np.stack(frames)).permute(3, 0, 1, 2)
    
    return video_tensor


In [10]:
model = Video3DCNN(num_classes=3)
print(model)

Video3DCNN(
  (conv1): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (bn1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (bn2): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (bn3): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=0, dilation=1, ceil_mode=False)
  (adaptive_pool): AdaptiveAvgPool3d(output_size=(1, 1, 1))
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_

In [13]:

data_dir = Path("../untracked")
video_path = data_dir / "head_cropped.mp4"


video_tensor = load_video_frames(video_path, max_frames=32)
batch = video_tensor.unsqueeze(0)  # Shape: (1, 3, 32, 112, 112)

model.eval()
with torch.no_grad():
    output = model(batch)
    
    probabilities = F.softmax(output, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1)

print(f"Model output shape: {output.shape}")
print(f"Predicted class: {predicted_class.item()}")
print(f"Class probabilities: {probabilities[0].tolist()}")

RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 3 is not equal to len(dims) = 4

In [ ]:

# Example of training setup (commented out)
"""
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop (simplified)
model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(batch)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
"""